In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from dermaml import model_setup

In [3]:
config_file = '/Users/ntin/DermaML/bin/config.yaml'
config = model_setup.read_config_yaml(config_file)

In [4]:
config.get('paths', {})['metadata_file']

'/Users/ntin/DermaML/data/2025-03-14-metadata_processed_test_split.csv'

In [ ]:
df = metadata_df
df.columns

m = model_setup.add_test_split_column(df.loc[df['gender_flag'] == str(1)])
f = model_setup.add_test_split_column(df.loc[df['gender_flag'] == str(0)])
y = model_setup.add_test_split_column(df.loc[df['young_old_flag'] == 0])
o = model_setup.add_test_split_column(df.loc[df['young_old_flag'] == 1])
l = model_setup.add_test_split_column(df.loc[df['hand'] == 'Left'])
r = model_setup.add_test_split_column(df.loc[df['hand'] == 'Right'])

In [ ]:
metadata_df = pd.read_csv(config.get('paths', {})['metadata_file'])
features_df = pd.read_csv(config.get('paths', {})['tabular_feature_file'])

# --- Prepare join key
metadata_df.loc[:,config['metadata_ref_header']] = (
    metadata_df[config['metadata_ref_header']].apply(
        lambda x:x[:-len(config['metadata_ref_extension'])]
))
# cross reference filenames
found_in_metadata = features_df[config['tabular_ref_header']].apply(
    lambda x: x in metadata_df[config['metadata_ref_header']].to_list()
)
# filter for rows found in metadata
features_df = features_df.loc[found_in_metadata]

# map ages to filenames
# features_df = features_df.drop(columns=config['tabular_ref_target'], inplace=False)
filename_age_map = dict(
    zip(
        metadata_df[config['metadata_ref_header']], 
        metadata_df[config['metadata_target']]
    )
)
mapped_ages = features_df[config['tabular_ref_header']].apply(
        lambda x: filename_age_map.get(x)
    )    
features_df.loc[:, config['metadata_target']] = mapped_ages

Xy = features_df.copy()
# -- Remove join keys
exclude_from_X = ['Unnamed: 0',
        config['metadata_ref_header'],
        config['tabular_ref_header'],
        config['tabular_ref_hand'],
        config['tabular_ref_target']
        ]
for col in exclude_from_X:
    if col in Xy.columns:
        Xy.drop(columns=[col],inplace=True)

_, test_indices = train_test_split(
    Xy.index, 
    test_size=0.7, 
    random_state=42    
)
Xy.loc[:, 'test_set'] = 0
Xy.loc[test_indices, 'test_set'] = 1

train = Xy.loc[Xy['test_set'] == 0]
test = Xy.loc[Xy['test_set'] == 1]

# keep only metadata filename and target column

# -- Construct DataFrame for model training and testing
# X = pd.merge(
#     left=metadata_df,
#     right=features_df,
#     left_on=header_features,
#     right_on=header_features,
    # how='inner',
# )

In [110]:
Xy.columns

Index(['relative_redness_mean', 'relative_redness_std', 'skin_folds_hessian',
       'skin_folds_hessian_pct_mask', 'lbp_0', 'lbp_1', 'lbp_2', 'lbp_3',
       'lbp_4', 'lbp_5', 'lbp_6', 'lbp_7', 'lbp_8', 'lbp_9', 'lbp_10',
       'contrast_scikit', 'correlation_scikit', 'energy_scikit',
       'homogeneity_scikit', 'age', 'test_set'],
      dtype='object')

In [102]:
features_df['age']

0      20
1      20
2      39
3      19
4      25
       ..
571    21
572    27
573    21
574    77
575    44
Name: age, Length: 572, dtype: int64

In [6]:
experiments = {
    'male': m,
    'female':f,
    'under_40':y,
    'over_40':o,
    'left':l,
    'right':r,
}
for k, v in experiments.items():
    csv_name = f'2025-03-15-NT-metadata_{k}_only_splitted.csv'
    # v.to_csv(csv_name)
    print(csv_name)

2025-03-15-NT-metadata_male_only_splitted.csv
2025-03-15-NT-metadata_female_only_splitted.csv
2025-03-15-NT-metadata_under_40_only_splitted.csv
2025-03-15-NT-metadata_over_40_only_splitted.csv
2025-03-15-NT-metadata_left_only_splitted.csv
2025-03-15-NT-metadata_right_only_splitted.csv
